# Tutorial: Estimating Social Contact Matrices with SocialMix

In [1]:
import numpy as np
import pandas as pd

from cntmosaic.utils import AgeBins
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import ParticipantGenerator, MatrixGenerator, ContactGenerator, Subgroup
from cntmosaic.vis import plot_mosaic, plot_mosaic_pixilated

from cntmosaic.models import SocialMix
from cntmosaic.analysis import ModelSummariserSocialMix, ModelEvaluatorSocialMix

import altair as alt

## 1. Introduction
The `SocialMix` class in `cntmosaic` is a Python implementation of the `socialmixr` algorithm for estimating social contact matrices from survey data. This tutorial will guide you through the process of using `SocialMix` to analyze contact survey data.

### Relationship to socialmixr
The original `socialmixr` is an R package widely used for estimating social contact matrices from survey data. The `SocialMix` class in `cntmosaic` replicates the core functionalities of `socialmixr`, allowing users to perform similar analyses within the Python ecosystem. This includes the ability to estimate contact intensities, apply reciprocity adjustments, compute contact rates, and bootstrap procedures for quantifying uncertainty. `SocialMix` also implements an adaptive age group merging strategy that automatically combines age groups with no participants with adjacent groups to ensure that the algorithm functions correctly even when certain age groups lack participant data.

### Mathematical Foundation
The core estimation follows these steps:

1. **Aggregate contacts by age group pairs**:
   $$
   y_{c,d} = \sum_{i = 1}^N 1_c(c_i) y_{i,d}
   $$
   where $1_c(c_i)$ is an indicator function that equals 1 if contact $i$ belongs to age group $c$, and $y_{i,d}$ is the number of contacts reported by participant $i$ with age group $d$.

2. **Compute contact intensity**:
   $$
   \hat{m}_{c,d} = \frac{y_{c,d}}{N_c}
   $$
   where $N_c$ is the number of participants in age group $c$.

3. **Apply reciprocity adjustment**:
   $$
   \hat{m}_{c,d}^{\dagger} = \frac{1}{2P_c} \left( \hat{m}_{c,d} P_c + \hat{m}_{d,c} P_d \right)
   $$
   ensuring that $P_c m_{c,d} = P_d m_{d,c}$ (total contact from $c$ to $d$ equals contacts from $d$ to $c$).

4. **Compute contact rates**:
   $$
   \hat{\gamma}_{c,d} = \frac{\hat{m}_{c,d}}{P_d}
   $$
   where $P_d$ is the population size of age group $d$.

## 2. Synthetic Data Generation
We'll use `cntmosaic`'s data generation tools to create realistic synthetic contact data. For details, refer to the "Tutorial: Generating Synthetic Contact Data".

We first load the contact pattern templates and population data:

In [2]:
templates = load_template_patterns(country='United_States')
df_age_dist = load_age_distribution(country='United_States')

age_dist = df_age_dist.P.values

We define a subgroup with 1000 participants with the `Subgroup` class:

In [3]:
my_population = Subgroup(
  n=1000,                       # Generate 1000 participants
  age_dist=age_dist,            # Use this age distribution
  mean_cint_margin=15.0,        # Average 15 contacts per person
  label='general'
)

We randomly simulate a contact intensity matrix using the `MatrixGenerator` class.

In [4]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
contact_matrix = matrix_gen.generate_single(
  subgroup=my_population,
  seed=42   # For reproducibility
)

print("Matrix shape:", contact_matrix.shape)
print("Average contacts per person:", contact_matrix.sum(axis=0).mean())

Matrix shape: (81, 81)
Average contacts per person: 14.999999999807594


Visualize the generated contact matrix with `plot_mosaic` function:

In [5]:
plot_mosaic(contact_matrix, title='Synthetic Contact Intensity Matrix')

alt.Chart(...)

The `ParticipantGenerator` class is then used to simulate participant data based on the defined subgroup.

In [6]:
# Initialize the participant generator
part_gen = ParticipantGenerator(my_population)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

   id  age_group
0   1         55
1   2         31
2   3         61
3   4         49
4   5          6


The `ContactGenerator` class simulates contact data for the participants:

In [7]:
cnt_gen = ContactGenerator(
  df_part=df_part,
  cint_matrices=contact_matrix,
  model='poisson'
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

Generated 12595 contacts
   id  age_cnt  y
0   1        5  1
1   1       10  1
2   1       20  2
3   1       27  1
4   1       36  1


Plot the empirical contact matrix from the simulated contact data:

In [8]:
emp_cint_matrix = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)
plot_mosaic(emp_cint_matrix, title='Empirical Contact Matrix', zlabel='Intensity')

alt.Chart(...)

## 3. Estimating Contact Matrices with SocialMix

### 3.1 Data Preparation

Besides the participant, contact, and population age distribution dataframes, the `SocialMix` class requires information about the age bins used for grouping the participants and contacts. We give these information in the form of an `AgeBins` object. Here, we define 5-year age bins from 0 to 80 years. The participant dataframe must also contain a column named `age_part` (or `age_grp_part` if age groups are pre-defined).

In [9]:
age_bins = AgeBins(0, 80, 5)
df_part['age_part'] = df_part['age_group']

### 3.2 Basic Matrix Estimation

In [10]:
# Initialize SocialMix
sm = SocialMix(
    df_part=df_part,
    df_cnt=df_cnt,
    df_age_dist=df_age_dist,
    age_bins=age_bins,
    symmetric=False,  # No reciprocity adjustment yet
    adaptive_merge=False,
    verbose=True
)

# Compute contact intensity matrix
M_est = sm.compute_cint()

print(f"Estimated matrix shape: {M_est.shape}")
print(f"Average contacts per person: {M_est.sum(axis=1).mean():.2f}")

Estimated matrix shape: (16, 16)
Average contacts per person: 15.10


#### Mathematical Details: Contact Intensity Estimation

The contact intensity matrix $M$ is computed as:
$$
M = N^{-1} Y
$$
where

- $Y \in \mathbb{R}^{B \times B}$ is the contact count matrix, with $Y_{c,d}$ representing the total number of contacts reported by participants in age group $c$ with contacts in age group $d$.
- $N \in \mathbb{R}^{B \times B}$ is a diagonal matrix where $N_{c,c}$ is the number of participants in age group $c$.
- $m_{c,d}$ represent the average number of contacts from age group $c$ to age group $d$.

In [11]:
# Plot the estimated contact intensity matrix
plot_mosaic_pixilated(
  M_est,
  age_bins,
  title="Estimated contact intensity matrix",
  zlabel="Intensity"
)

alt.Chart(...)

### 3.3 Contact Rate Matrix

In [12]:
# Compute contact rate matrix
rate_est = sm.compute_rate()

#### Mathematical Details: Contact Rates

The contact rate normalizes by target population size:
$$
\Gamma = M P^{-1}
$$
where $P = \text{diag}(P_1, \ldots, P_B)$ contains population sizes.

In [13]:
# Plot the estimated contact rate matrix
plot_mosaic_pixilated(
  rate_est,
  age_bins,
  title="Estimated contact rate matrix",
  zlabel="Rate"
)

alt.Chart(...)

### 3.4 Reciprocity Adjustment

In [14]:
# Apply symmetric adjustment
sm = SocialMix(
    df_part=df_part,
    df_cnt=df_cnt,
    df_age_dist=df_age_dist,
    age_bins=age_bins,
    symmetric=True,
    adaptive_merge=False,
    verbose=True
)

# Compute contact intensity matrix with reciprocity adjustment
M_sym = sm.compute_cint()

#### Mathematical Details: Reciprocity

The reciprocity condition ensures that the total contacts are equal:
$$
P_c m_{c,d} = P_d m_{d,c}
$$
This is enforced through the symmetric adjustment:
$$
m^\dagger_{c,d} = \frac12 \left( m_{c,d} + \frac{P_d}{P_c} m_{d,c} \right)
$$

**Why this matters**: Survey data may have asymmetric reporting. Reciprocity adjustment corrects this using population weights.

In [15]:
# Compute contact rate matrix with reciprocity adjustment
rate_est_sym = sm.compute_rate()

In [16]:
plt_non_sym = plot_mosaic_pixilated(
  rate_est,
  age_bins,
  title="Estimated contact rate matrix (non-symmetric)",
  zlabel="Rate"
)

plt_sym = plot_mosaic_pixilated(
  rate_est_sym,
  age_bins,
  title="Estimated rate pattern (symmetric)",
  zlabel="Rate"
)

alt.hconcat(plt_non_sym, plt_sym)

alt.HConcatChart(...)

### 3.5 Uncertainty Quantification via Bootstrapping

The uncertainty in the estimated contact matrices can be quantified using a bootstrap procedure. This involves resampling the participant data with replacement and re-estimating the contact matrices multiple times to generate a distribution of estimates. From this distribution, confidence intervals and other uncertainty measures can be derived.

In [17]:
boot = sm.run_bootstrap(n_boot=500, random_state=42)

Bootstrapping: 100%|██████████| 500/500 [00:54<00:00,  9.12it/s]


### 3.6 Summarizing Bootstrap Results
Bootstrap results can be summarized using the `ModelSummariserSocialMix` class. This class provides methods to extract key statistics from the bootstrap samples, such as mean estimates, confidence intervals, and other relevant metrics.

In [18]:
summariser = ModelSummariserSocialMix(sm)
summary = summariser.summarise_cint(alpha=0.05)

In [19]:
plt_lower = plot_mosaic_pixilated(summary['lower'], age_bins, title='Lower', zlabel='Intensity')
plt_median = plot_mosaic_pixilated(summary['median'], age_bins, title='Median', zlabel='Intensity')
plt_upper = plot_mosaic_pixilated(summary['upper'], age_bins, title='Upper', zlabel='Intensity')

alt.hconcat(plt_lower, plt_median, plt_upper)

alt.HConcatChart(...)

### 3.7 Evaluating Model Accuracy
The accuracy of the estimated contact matrices can be evaluated by comparing them to the true contact matrices used in the synthetic data generation. The `ModelEvaluatorSocialMix` class can be used to compute a wide range of accuracy metrics.

In [20]:
# Initialize evaluator
evaluator = ModelEvaluatorSocialMix(summariser, cint_matrix_true=contact_matrix)

In [21]:
# Default metrics
evaluator.evaluate()

,alpha,discretization_error,estimation_error,total_error,interval_score,coverage
0,0.05,0.033214,0.008119,0.0335,1.094262,77.793019


In [22]:
# Extended metrics
evaluator.evaluate(include_extended=True)

,alpha,discretization_error,estimation_error,total_error,interval_score,coverage,rmse_total,rmse_estimation,rmse_discretization,mae_total,mae_estimation,relative_error
0,0.05,0.033214,0.008119,0.0335,1.094262,77.793019,0.183031,0.090107,0.182246,0.045859,0.064699,0.506824
